The sklearn baseline and data checks live in `src/feedbio` and run on a laptop. This notebook is the Colab QLoRA path: 4-bit Mistral-7B + LoRA on `prompt.jsonl`.

Needs a GPU, `HF_TOKEN` in Colab secrets, and `prompt.jsonl` in the working directory.

In [ ]:
!pip install -q transformers==4.44.2 datasets==3.0.1 peft==0.13.0 accelerate==1.0.1 bitsandbytes==0.44.1


In [ ]:
import transformers, peft
print(transformers.__version__, peft.__version__)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2",revision="main", use_fast=False)
print("Tokenizer loaded successfully.")


In [6]:
from huggingface_hub import login
from google.colab import userdata

# Login with your Hugging Face token (make sure to rotate this if this notebook gets shared)
hf_token = userdata.get('HF_TOKEN')
login(hf_token)



In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2",revision="main", use_fast=False)
tokenizer.pad_token = tokenizer.eos_token  # Mistral uses EOS as padding

# Set 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="bfloat16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    quantization_config=bnb_config,
    device_map="auto"
)


In [ ]:
from datasets import load_dataset

# Load your dataset from the ChatML-formatted JSONL file
dataset = load_dataset("json", data_files="prompt.jsonl")["train"]

# Seeded split. This is still row-wise, so repeated questions can leak
# across the split. The sklearn baseline uses a question-level holdout instead.
dataset = dataset.train_test_split(test_size=0.1, seed=42)


In [10]:
def tokenize(example):
    chat_text = tokenizer.apply_chat_template(example["messages"], tokenize=False)

    tokenized = tokenizer(
        chat_text,
        padding="max_length",        # pad all to same length
        truncation=True,             # truncate if too long
        max_length=1024,             # keep it safe for Mistral
        return_tensors="pt"          # force PyTorch tensors
    )

    return {
        "input_ids": tokenized["input_ids"][0],
        "attention_mask": tokenized["attention_mask"][0],
        # Full-sequence labels: loss includes the question, not just the mark.
        # Assistant-only masking would be better; left as a follow-up.
        "labels": tokenized["input_ids"][0].clone()
    }


In [ ]:
tokenized_train = dataset["train"].map(tokenize, remove_columns=dataset["train"].column_names)
tokenized_eval = dataset["test"].map(tokenize, remove_columns=dataset["test"].column_names)


In [12]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments

# Define LoRA configuration (only train a few adapter layers)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to your Mistral model
model = get_peft_model(model, lora_config)
# Define training arguments
training_args = TrainingArguments(
    output_dir="./mistral-bio-checkpoints",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=15,
    learning_rate=2e-4,
    save_steps=200,
    logging_steps=1,
    bf16=True,
    save_total_limit=2,
    do_eval=True,
    eval_steps=50,
    seed=42,
    deepspeed=None
)

In [ ]:

from transformers import Trainer
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We’re training a causal LM like Mistral
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    tokenizer=tokenizer
)

trainer.train()


In [14]:
model_path = "./feedbio"

# Save the model
model.save_pretrained(model_path)

# Save the tokenizer
tokenizer.save_pretrained(model_path)


('./feedbio/tokenizer_config.json',
 './feedbio/special_tokens_map.json',
 './feedbio/chat_template.jinja',
 './feedbio/tokenizer.model',
 './feedbio/added_tokens.json')

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Copy the folder to your Drive
import shutil
import os

source = "./feedbio"
destination = "/content/drive/My Drive/feedbio_final_model"

# Check if it already exists to avoid errors
if os.path.exists(destination):
    shutil.rmtree(destination)

shutil.copytree(source, destination)
print(f"✅ Success! Saved to Google Drive at: {destination}")